# Notebook 04 — Embeddings for DocuTrace

**Goal:** understand how text is represented as vectors, generate an embedding for every chunk, compare vectors, embed a user query, and retrieve the most similar chunks.

**Prerequisite:** complete Notebook 03 and create `data/processed/chunks_recursive_experiment.json`.

## Learning outcomes

By the end, you should be able to explain:
- what an embedding is and why RAG uses it;
- what vector dimensions mean (and what they do *not* mean);
- why one vector is created for each chunk;
- how cosine similarity compares vectors;
- how a user question is embedded using the same model;
- why a similarity score is a ranking signal, not proof that an answer is correct;
- how to save vectors while keeping their chunk IDs and source metadata aligned.

> **Model for this baseline experiment:** `sentence-transformers/all-MiniLM-L6-v2`. It produces 384-dimensional vectors. We are using it as a compact learning baseline, not declaring it the final production model. Its model card notes that inputs longer than 256 word pieces are truncated, so chunk length matters.


## 1. Install the embedding library

Open the VS Code PowerShell terminal, ensure the DocuTrace virtual environment is active (`(.venv)` appears in the prompt), and run:

```powershell
python -m pip install -U sentence-transformers
```

This package uses PyTorch under the hood. Installation can take a little time because PyTorch is a larger dependency than our earlier PDF packages. We will run this compact embedding model on **CPU**; this is not a large generative language model.

After installation, return to this notebook and restart the kernel if VS Code asks you to. Confirm that the selected kernel is **Python (DocuTrace)**.

Official references:
- https://sbert.net/docs/sentence_transformer/usage/usage.html
- https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2


In [1]:
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

print("NumPy:", np.__version__)
print("Sentence Transformers imported successfully.")

c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.5.4
Sentence Transformers imported successfully.


## 2. Load our chunk records

We embed **chunks**, not entire PDFs. Each chunk is a small text unit that can later be retrieved independently.

Our chunk records also contain metadata such as `chunk_id`, `source`, `page_number`, and `text`. Keep that metadata: the vector helps us find a chunk, while the metadata tells us where the chunk came from.

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

chunks_path = PROJECT_ROOT / "data" / "processed" / "chunks_recursive_experiment.json"

if not chunks_path.exists():
    raise FileNotFoundError(
        f"Could not find {chunks_path}. Complete Notebook 03 first."
    )

with chunks_path.open("r", encoding="utf-8") as file:
    chunk_records = json.load(file)

if not isinstance(chunk_records, list) or not chunk_records:
    raise ValueError("The chunk file is empty or does not contain a list.")

required_fields = {"chunk_id", "source", "page_number", "text"}
for index, record in enumerate(chunk_records):
    missing = required_fields - record.keys()
    if missing:
        raise ValueError(f"Chunk {index} is missing fields: {sorted(missing)}")
    if not isinstance(record["text"], str) or not record["text"].strip():
        raise ValueError(f"Chunk {index} has empty or invalid text.")

print("Chunks loaded:", len(chunk_records))
print("First chunk ID:", chunk_records[0]["chunk_id"])
print("First chunk source:", chunk_records[0]["source"])
print("First chunk page:", chunk_records[0]["page_number"])
print("First chunk text:\n", chunk_records[0]["text"])

Chunks loaded: 12
First chunk ID: data_pipeline_runbook::p1::c001
First chunk source: data_pipeline_runbook.pdf
First chunk page: 1
First chunk text:
 DATA PIPELINE RUNBOOK
Pipeline: CUSTOMER_DAILY_LOAD
Schedule: Daily at 02:00 UTC.
Upstream dependency: CUSTOMER_RAW_INGEST.
Normal operation:


## 3. Understand the embedding model

An **embedding** is a list (vector) of numbers generated from an input. A model learns to place texts with related meanings in nearby regions of a vector space.

Important distinctions:

- The embedding model is **not** the LLM that writes the final answer.
- The vector dimensions are learned numerical features. We generally cannot label dimension 17 as "database" or dimension 42 as "failure".
- A vector is not a summary or a human-readable translation of the text.
- The model must be used consistently for both document chunks and user queries. Changing the embedding model generally requires re-embedding the stored chunks.

We choose `all-MiniLM-L6-v2` because it is a compact sentence/paragraph embedding baseline. It produces 384 numbers for each input. Its model card says inputs longer than 256 word pieces are truncated; do not assume a long chunk is fully represented.

In [4]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# CPU keeps this experiment independent of a CUDA/GPU setup.
model = SentenceTransformer(MODEL_NAME, device="cpu")

embedding_dimension = model.get_embedding_dimension()
print("Model:", MODEL_NAME)
print("Embedding dimension:", embedding_dimension)
print("Model maximum sequence length:", model.max_seq_length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 175.81it/s]


Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Model maximum sequence length: 256


## 4. A small embedding demonstration

Before embedding all our DocuTrace chunks, compare a few example sentences.

The sentences below discuss a similar operational issue using different wording. We expect their semantic representations to be relatively close compared with a clearly unrelated sentence. The exact scores are model-dependent; do not memorize a particular number.

In [5]:
demo_sentences = [
    "The upstream partition is missing, so the customer load cannot proceed.",
    "The required business-date data has not arrived from the source job.",
    "Check the Hive table partitions before running a query.",
    "A sunny afternoon is a good time to go for a walk.",
]

demo_vectors = model.encode(
    demo_sentences,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print("Number of sentences:", len(demo_sentences))
print("Vector matrix shape:", demo_vectors.shape)
print("First vector (first 10 values):\n", demo_vectors[0][:10])

Number of sentences: 4
Vector matrix shape: (4, 384)
First vector (first 10 values):
 [-0.0588598  -0.01371577  0.03118551 -0.01527905  0.01098585 -0.02963707
 -0.07362263  0.00292239  0.00249017 -0.00465388]


### How to read the shape

If the output shape is `(4, 384)`:

- `4` means four sentences were encoded;
- `384` means each sentence is represented by 384 numbers.

The same model gives the same vector dimension for every input. We will use the same idea for our chunk records.

In [6]:
# A quick check: each normalized vector should have a length close to 1.
vector_norms = np.linalg.norm(demo_vectors, axis=1)
print("Vector norms:", np.round(vector_norms, 4))

Vector norms: [1. 1. 1. 1.]


## 5. Similarity: compare meanings as vectors

One common comparison is **cosine similarity**:

`cosine_similarity(A, B) = (A · B) / (||A|| × ||B||)`

- `A · B` is the dot product of the two vectors.
- `||A||` and `||B||` are their lengths (norms).
- With normalized vectors, the dot product equals cosine similarity.

Cosine similarity is commonly interpreted as higher = more similar direction. It is not a probability, and there is no universal score threshold that proves two texts are relevant.

In [7]:
# All demo vectors were normalized above, so their dot products
# are cosine similarities.
demo_similarity = demo_vectors @ demo_vectors.T

print("Cosine similarity matrix (rounded):")
print(np.round(demo_similarity, 3))

print("\nSentence order:")
for index, sentence in enumerate(demo_sentences):
    print(f"{index}: {sentence}")

Cosine similarity matrix (rounded):
[[ 1.     0.331  0.365 -0.046]
 [ 0.331  1.     0.243  0.019]
 [ 0.365  0.243  1.    -0.033]
 [-0.046  0.019 -0.033  1.   ]]

Sentence order:
0: The upstream partition is missing, so the customer load cannot proceed.
1: The required business-date data has not arrived from the source job.
2: Check the Hive table partitions before running a query.
3: A sunny afternoon is a good time to go for a walk.


### Interpret the similarity matrix

The matrix compares each sentence with every other sentence. The diagonal is approximately `1.0`, because each sentence is being compared with itself.

Look at the scores between the first two sentences and compare them with the first and fourth. Your model's actual values may vary slightly with package/model versions. The goal is to understand relative similarity, not to treat a score as a guaranteed relevance label.

## 6. Generate one embedding for each DocuTrace chunk

Now we embed every chunk from `chunks_recursive_experiment.json`.

For retrieval systems, Sentence Transformers provides `encode_document()` for content being indexed and `encode_query()` for the user's search query. Some models apply different prompts to queries and documents; models without those special prompts behave like ordinary encoding. We will use these task-specific methods because they make the intended roles explicit.

In [8]:
chunk_texts = [record["text"] for record in chunk_records]

chunk_embeddings = model.encode_document(
    chunk_texts,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Number of chunks:", len(chunk_texts))
print("Embedding matrix shape:", chunk_embeddings.shape)
print("Expected shape:", (len(chunk_records), embedding_dimension))

assert chunk_embeddings.shape == (len(chunk_records), embedding_dimension)
assert np.isfinite(chunk_embeddings).all()
print("All chunks have a valid embedding.")

Batches: 100%|██████████| 2/2 [00:00<00:00,  8.56it/s]

Number of chunks: 12
Embedding matrix shape: (12, 384)
Expected shape: (12, 384)
All chunks have a valid embedding.


### Understand the matrix

If there are 12 chunks and 384 dimensions, the shape should be `(12, 384)`:

- row 1 = embedding for chunk 1;
- row 2 = embedding for chunk 2;
- and so on;
- each row has 384 floating-point values.

**Ordering matters:** row `i` belongs to `chunk_records[i]`. We will save the chunk IDs alongside the vectors so that this mapping stays explicit.

In [9]:
chunk_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Minimum vector norm:", round(float(chunk_norms.min()), 4))
print("Maximum vector norm:", round(float(chunk_norms.max()), 4))

print("\nExample: first chunk and its first 10 vector values")
print("Chunk ID:", chunk_records[0]["chunk_id"])
print(np.round(chunk_embeddings[0][:10], 4))

Minimum vector norm: 1.0
Maximum vector norm: 1.0

Example: first chunk and its first 10 vector values
Chunk ID: data_pipeline_runbook::p1::c001
[-0.0607  0.0117 -0.0068  0.0185 -0.056  -0.0851 -0.0801  0.0152 -0.0366
 -0.0443]


## 7. Embed a user question and retrieve similar chunks

This is the first miniature semantic retrieval experiment. We have not built a vector database yet. For now, NumPy will compare the query vector with every chunk vector in memory.

The user's question must be embedded with the **same model** used to embed the chunks.

In [10]:
query = "What should I investigate if the customer daily load fails because an upstream partition is missing?"

query_embedding = model.encode_query(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

# Because both query and chunk vectors are normalized,
# matrix multiplication gives cosine similarity scores.
scores = chunk_embeddings @ query_embedding

ranked_indices = np.argsort(scores)[::-1]

print("Question:", query)
print("\nTop matching chunks:")

for rank, index in enumerate(ranked_indices[:5], start=1):
    record = chunk_records[int(index)]
    print(f"\nRank: {rank} | Similarity: {float(scores[index]):.4f}")
    print("Chunk ID:", record["chunk_id"])
    print("Source:", record["source"])
    print("Page:", record["page_number"])
    print("Text:", record["text"])



Question: What should I investigate if the customer daily load fails because an upstream partition is missing?

Top matching chunks:

Rank: 1 | Similarity: 0.7487
Chunk ID: data_pipeline_runbook::p2::c001
Source: data_pipeline_runbook.pdf
Page: 2
Text: DATA PIPELINE RUNBOOK - INCIDENT RESPONSE
Symptom: CUSTOMER_DAILY_LOAD failed because an upstream
partition is missing.
Investigation:
1. Check the upstream job status and logs.

Rank: 2 | Similarity: 0.6035
Chunk ID: hive_troubleshooting_guide::p1::c002
Source: hive_troubleshooting_guide.pdf
Page: 1
Text: 2. Run SHOW PARTITIONS on the table.
3. Check the partition column and date format.
4. Verify that the upstream load completed successfully.

Rank: 3 | Similarity: 0.5818
Chunk ID: data_pipeline_runbook::p2::c002
Source: data_pipeline_runbook.pdf
Page: 2
Text: 2. Confirm the expected business date and partition format.
3. Verify the upstream ingestion completed.
Recovery:
Retry the downstream job only after the upstream partition

Rank

### What just happened?

1. We turned the user's question into a 384-dimensional query vector.
2. We compared it with each stored chunk vector.
3. We sorted the scores from highest to lowest.
4. We displayed the top five chunks with their original source metadata.

This is the basic idea of **dense retrieval**. We will later compare it with keyword search (BM25), hybrid retrieval, metadata filters, and reranking.

A high similarity score does not guarantee that a chunk contains the full answer. Retrieval finds candidate evidence; the later generation stage must use that evidence carefully.

## 8. Save embeddings and preserve their mapping

We'll save:

- a compressed NumPy file containing the embedding matrix and corresponding chunk IDs;
- a JSON information file describing the model and experiment.

The original chunk text and source/page metadata remain in `chunks_recursive_experiment.json`. The saved chunk IDs allow us to connect each vector back to its original record.

In [11]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

embeddings_path = processed_dir / "chunk_embeddings_minilm.npz"
metadata_path = processed_dir / "chunk_embeddings_minilm_metadata.json"

chunk_ids = np.array([record["chunk_id"] for record in chunk_records])

np.savez_compressed(
    embeddings_path,
    embeddings=chunk_embeddings,
    chunk_ids=chunk_ids,
)

embedding_metadata = {
    "model_name": MODEL_NAME,
    "embedding_dimension": int(embedding_dimension),
    "number_of_chunks": len(chunk_records),
    "normalized_embeddings": True,
    "source_chunk_file": chunks_path.name,
    "similarity_method_used_in_experiment": "cosine similarity (dot product of normalized vectors)",
}

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(embedding_metadata, file, ensure_ascii=False, indent=2)

print("Saved embeddings:", embeddings_path)
print("Saved metadata:", metadata_path)

Saved embeddings: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag\data\processed\chunk_embeddings_minilm.npz
Saved metadata: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag\data\processed\chunk_embeddings_minilm_metadata.json


## 9. Verify that saved vectors can be loaded

A file being saved is not enough. Reload it and confirm its shape and chunk ID mapping.

In [12]:
with np.load(embeddings_path, allow_pickle=False) as saved:
    reloaded_embeddings = saved["embeddings"]
    reloaded_chunk_ids = saved["chunk_ids"]

assert reloaded_embeddings.shape == chunk_embeddings.shape
assert np.array_equal(reloaded_chunk_ids, chunk_ids)
assert np.allclose(reloaded_embeddings, chunk_embeddings)

print("Reload verification passed.")
print("Reloaded matrix shape:", reloaded_embeddings.shape)
print("First saved chunk ID:", reloaded_chunk_ids[0])

Reload verification passed.
Reloaded matrix shape: (12, 384)
First saved chunk ID: data_pipeline_runbook::p1::c001


## 10. Knowledge check

Try answering these in your own words before looking up the answers:

1. What is an embedding? How is it different from the original text?
2. If we have 12 chunks and a 384-dimensional model, why is the matrix shape `(12, 384)`?
3. Why should the query and document chunks use the same embedding model?
4. What does cosine similarity measure? Why is it not a probability?
5. Why do we save chunk IDs alongside the embedding matrix?
6. Does a high similarity score guarantee that the generated answer is correct? Why not?
7. What could happen if a chunk exceeds the model's maximum input length?

## Completion checklist

- [ ] Installed `sentence-transformers` into the DocuTrace `.venv`.
- [ ] Loaded the recursive chunk records.
- [ ] Generated embeddings and checked their dimensions.
- [ ] Compared sentence and chunk vectors with cosine similarity.
- [ ] Embedded a question and inspected the top matching chunks.
- [ ] Saved and reloaded the embeddings successfully.

## What comes next

**Notebook 05 — Vector stores and indexing.** We will first understand why a NumPy matrix is enough for this tiny experiment, then learn what a vector database/index contributes: storage, efficient nearest-neighbor search, metadata association, and retrieval at larger scale. We will not add a vector database until the vector and similarity concepts are clear.